# Audiobook AI - Local Runner

Notebook này chạy local theo kiến trúc:

`Streamlit UI -> FastAPI API -> SQLite Queue -> XTTS Pipeline -> Download audio`

Chạy notebook từ repo local. FastAPI dùng `http://127.0.0.1:8000`, Streamlit dùng `http://127.0.0.1:8501`.

## 0. Chuẩn bị local

- Cài Python >= 3.10.
- Cài `ffmpeg` trên máy: `apt install ffmpeg`, `brew install ffmpeg`, hoặc `choco install ffmpeg`.
- Nếu dùng XTTS GPU, cần CUDA/Torch tương thích và GPU khả dụng.
- Nếu repo Hugging Face private/gated, đặt biến môi trường `HF_TOKEN` trước khi chạy notebook.

In [ ]:
# [1] Xác định repo root
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'app.py').exists() and (candidate / 'streamlit_app.py').exists():
            return candidate
    raise RuntimeError('Không tìm thấy repo root chứa src/app.py và streamlit_app.py')

PROJECT_DIR = find_repo_root(Path.cwd())
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR =', PROJECT_DIR)
print('Python =', sys.version)


In [ ]:
# [2] Kiểm tra GPU và ffmpeg
import shutil
import subprocess
import sys

print('ffmpeg =', shutil.which('ffmpeg') or 'NOT FOUND')
if not shutil.which('ffmpeg'):
    print('Cần cài ffmpeg trước khi xuất mp3 hoặc xử lý audio.')

try:
    import torch
    print('Torch =', torch.__version__)
    print('CUDA available =', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU =', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


In [ ]:
# [3] Cài/cập nhật dependencies local
# Nếu bạn đang dùng virtualenv/conda, hãy chọn đúng kernel trước khi chạy cell này.
import subprocess
import sys
from pathlib import Path

XTTS_RUNTIME_REPO = 'https://github.com/nguyenhoanganh2002/XTTSv2-Finetuning-for-New-Languages.git'
XTTS_RUNTIME_DIR = Path('models/XTTSv2-Finetuning-for-New-Languages')

if not XTTS_RUNTIME_DIR.exists():
    subprocess.run(['git', 'clone', XTTS_RUNTIME_REPO, str(XTTS_RUNTIME_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(XTTS_RUNTIME_DIR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-multipart', 'huggingface_hub', 'hf_transfer', 'torchcodec'], check=True)

import fastapi, streamlit
print('Dependencies OK')


In [24]:
# [4] Tải model XTTS và tạo .env local
from pathlib import Path
import os
import sys

from huggingface_hub import snapshot_download


def has_xtts_model_files(model_dir: Path) -> bool:
    return (
        model_dir.exists()
        and (model_dir / 'config.json').exists()
        and (model_dir / 'vocab.json').exists()
        and any(model_dir.glob('*.pth'))
    )

XTTS_HF_REPO = os.getenv('XTTS_HF_REPO', 'aiMy144/XTTSv2VietAudiobook')
DEFAULT_MODEL_DIR = PROJECT_DIR / 'model'
SCRIPTS_MODEL_DIR = PROJECT_DIR / 'scripts/model'
XTTS_LOCAL_DIR = Path(os.getenv('XTTS_MODEL_NAME_OR_PATH') or '') if os.getenv('XTTS_MODEL_NAME_OR_PATH') else None

if XTTS_LOCAL_DIR is None:
    XTTS_LOCAL_DIR = SCRIPTS_MODEL_DIR if has_xtts_model_files(SCRIPTS_MODEL_DIR) else DEFAULT_MODEL_DIR
XTTS_LOCAL_DIR = XTTS_LOCAL_DIR.resolve()

XTTS_RUNTIME_DIR = PROJECT_DIR / 'models/XTTSv2-Finetuning-for-New-Languages'
HF_TOKEN = os.getenv('HF_TOKEN') or None

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

if not has_xtts_model_files(XTTS_LOCAL_DIR):
    snapshot_download(
        repo_id=XTTS_HF_REPO,
        repo_type='model',
        local_dir=str(XTTS_LOCAL_DIR),
        token=HF_TOKEN,
        max_workers=8,
    )

if str(XTTS_RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(XTTS_RUNTIME_DIR))

from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

STORAGE_DIR = PROJECT_DIR / 'storage'
VOICE_DIR = PROJECT_DIR / 'data/voice_samples'
STORAGE_DIR.mkdir(exist_ok=True)
VOICE_DIR.mkdir(parents=True, exist_ok=True)

env_text = f'''API_BASE_URL=http://127.0.0.1:8000
CORS_ORIGINS=http://localhost:8501,http://127.0.0.1:8501
STORAGE_DIR={STORAGE_DIR}
MAX_UPLOAD_MB=200
MAX_CONCURRENT_JOBS=1
TTS_ENGINE=xtts_gpu
XTTS_MODEL_NAME_OR_PATH={XTTS_LOCAL_DIR}
XTTS_CONFIG_PATH=
XTTS_VOCAB_PATH=
XTTS_RUNTIME_DIR={XTTS_RUNTIME_DIR}
XTTS_VOICE_DIR={VOICE_DIR}
HF_TOKEN={HF_TOKEN or ''}
'''
(PROJECT_DIR / '.env').write_text(env_text, encoding='utf-8')
print('.env created at', PROJECT_DIR / '.env')
print('XTTS model dir =', XTTS_LOCAL_DIR)
print('Voice dir =', VOICE_DIR)


.env created at /workspace/CSC15012-Final-Project-Audiobook-Generation-System/.env
XTTS model dir = /workspace/CSC15012-Final-Project-Audiobook-Generation-System/scripts/model
Voice dir = /workspace/CSC15012-Final-Project-Audiobook-Generation-System/data/voice_samples


## 5. Reference Voices

XTTS cần ít nhất một file `.wav` trong `data/voice_samples`. Repo có sẵn vài file mẫu; có thể thêm file WAV khác vào folder này.

In [ ]:
# [5] Kiểm tra reference voices
from pathlib import Path

voice_dir = PROJECT_DIR / 'data/voice_samples'
voices = sorted(p.name for p in voice_dir.glob('*.wav'))
print(f'Found {len(voices)} reference voices in {voice_dir}')
for name in voices:
    print('-', name)

assert voices, f'Thiếu reference voice WAV trong {voice_dir}'


In [52]:
# [6] Chạy FastAPI local
import os
import signal
import socket
import subprocess
import sys
import threading
import time
from pathlib import Path

API_PORT = 8000
API_BASE_URL = f'http://127.0.0.1:{API_PORT}'
api_log = []
api_processes = globals().setdefault('api_processes', [])


def stop_processes(processes):
    for proc in list(processes):
        if proc.poll() is not None:
            continue
        try:
            if os.name == 'nt':
                proc.terminate()
            else:
                os.killpg(proc.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except Exception:
            proc.terminate()
    for proc in list(processes):
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
    processes.clear()


def ready(port):
    try:
        socket.create_connection(('127.0.0.1', port), timeout=1).close()
        return True
    except OSError:
        return False


def resolve_xtts_model_dir() -> Path:
    env_model = os.getenv('XTTS_MODEL_NAME_OR_PATH')
    if env_model:
        return Path(env_model).resolve()
    if 'XTTS_LOCAL_DIR' in globals():
        return Path(XTTS_LOCAL_DIR).resolve()
    scripts_model = PROJECT_DIR / 'scripts/model'
    if scripts_model.exists() and any(scripts_model.glob('*.pth')):
        return scripts_model.resolve()
    return (PROJECT_DIR / 'model').resolve()

stop_processes(api_processes)

def start_api():
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_DIR) + os.pathsep + env.get('PYTHONPATH', '')
    env['API_BASE_URL'] = API_BASE_URL
    env['TTS_ENGINE'] = 'xtts_gpu'
    env['STORAGE_DIR'] = str(PROJECT_DIR / 'storage')
    env['XTTS_MODEL_NAME_OR_PATH'] = str(resolve_xtts_model_dir())
    env['XTTS_RUNTIME_DIR'] = str(PROJECT_DIR / 'models/XTTSv2-Finetuning-for-New-Languages')
    env['XTTS_VOICE_DIR'] = str(PROJECT_DIR / 'data/voice_samples')

    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == 'nt' else 0
    p = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'src.app:app', '--host', '127.0.0.1', '--port', str(API_PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        cwd=PROJECT_DIR,
        start_new_session=(os.name != 'nt'),
        creationflags=creationflags,
    )
    api_processes.append(p)
    for line in p.stdout:
        api_log.append(line.rstrip())

threading.Thread(target=start_api, daemon=True).start()

print('XTTS model dir =', resolve_xtts_model_dir())
print('Waiting for FastAPI', end='')
for _ in range(120):
    time.sleep(1)
    print('.', end='', flush=True)
    if ready(API_PORT):
        break
print()

assert ready(API_PORT), 'FastAPI chưa lên. Chạy cell logs bên dưới để xem lỗi.'
print('FastAPI local OK:', API_BASE_URL)


XTTS model dir = /workspace/CSC15012-Final-Project-Audiobook-Generation-System/scripts/model
Waiting for FastAPI...
FastAPI local OK: http://127.0.0.1:8000


In [53]:
# [7] Chạy Streamlit local
import os
import signal
import socket
import subprocess
import sys
import threading
import time

UI_PORT = 8501
UI_URL = f'http://127.0.0.1:{UI_PORT}'
ui_log = []
ui_processes = globals().setdefault('ui_processes', [])

stop_processes(ui_processes)

def start_streamlit():
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_DIR) + os.pathsep + env.get('PYTHONPATH', '')
    env['API_BASE_URL'] = API_BASE_URL
    env['STREAMLIT_BROWSER_GATHER_USAGE_STATS'] = 'false'
    env['STREAMLIT_GLOBAL_SHOW_WARNING_ON_DIRECT_EXECUTION'] = 'false'

    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == 'nt' else 0
    p = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'streamlit_app.py',
         f'--server.port={UI_PORT}', '--server.headless=true',
         '--server.enableCORS=false', '--server.enableXsrfProtection=false',
         '--browser.gatherUsageStats=false'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        cwd=PROJECT_DIR,
        start_new_session=(os.name != 'nt'),
        creationflags=creationflags,
    )
    ui_processes.append(p)
    for line in p.stdout:
        ui_log.append(line.rstrip())

threading.Thread(target=start_streamlit, daemon=True).start()

print('Waiting for Streamlit', end='')
for _ in range(60):
    time.sleep(1)
    print('.', end='', flush=True)
    if ready(UI_PORT):
        break
print()

assert ready(UI_PORT), 'Streamlit chưa lên. Chạy cell logs bên dưới để xem lỗi.'
print('=' * 80)
print('OPEN STREAMLIT:', UI_URL)
print('FastAPI:', API_BASE_URL)
print('=' * 80)


Waiting for Streamlit.
OPEN STREAMLIT: http://127.0.0.1:8501
FastAPI: http://127.0.0.1:8000


In [54]:
# [8] Xem logs khi cần debug
print('=== API LOG ===')
print('\n'.join(api_log[-200:]) or '(empty)')
print('\n=== STREAMLIT LOG ===')
print('\n'.join(ui_log[-200:]) or '(empty)')


=== API LOG ===
INFO:     Started server process [33034]
INFO:     Waiting for application startup.
[VoiceAgent] Connecting to Qdrant at data/qdrant_voice_db...
[VoiceAgent] Connected to collection 'voices'.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
[Summarizer] Map Phase: 4 chunks...
  -> Chunk 1/4 done
  -> Chunk 2/4 done
  -> Chunk 3/4 done
  -> Chunk 4/4 done
[Summarizer] Reduce Phase: aggregating...
[VoiceAgent] Voice 'child_voice_01' has no WAV sample. Falling back to 'female_hn_01'.
[VoiceAgent] Vector match: female_hn_01 (score: 0.052) for context: 'Tóm tắt: Bộ Quốc phòng Pháp đã tổ chức cuộc tấn cô...'
[VoiceAgent] Vector match: female_hcm_02 (score: 0.053) for context: 'Nhân vật: đại đội. Cảm xúc chủ đạo: Chiêm nghiệm /...'
[VoiceAgent] Vector match: male_hn_01 (score: 0.043) for context: 'Nhân vật: Tiểu đoàn. Cảm xúc chủ đạo: Chiêm nghiệm...'
[VoiceAgent] Vector match: female_hcm_02 (score: 0.018) for co

In [51]:
# [9] Dừng FastAPI / Streamlit khi cần
for processes in (globals().get('api_processes', []), globals().get('ui_processes', [])):
    if 'stop_processes' in globals():
        stop_processes(processes)

print('Stopped FastAPI and Streamlit.')


Stopped FastAPI and Streamlit.


## Cách test local

1. Chạy lần lượt các cell `[1]` đến `[7]`.
2. Mở `http://127.0.0.1:8501` trong trình duyệt.
3. Upload EPUB, chọn định dạng, rồi bấm `Create audiobook job`.
4. Nếu lỗi, chạy cell `[8]` để xem API và Streamlit logs.